1.	Hazırlık ve Ortam Ayarları
"""Analiz için gerekli kütüphaneler (pandas, numpy, matplotlib, seaborn, scipy) nasıl içe aktarılır?
 Grafiklerin görsel stili (tema, renk paleti, boyut) nasıl standart hale getirilir?
 Sonuçların tekrarlanabilir olması için rastgelelik nasıl sabitlenir (random_state/seed)?
 Pandas'ta tüm sütun/satırların ve ondalık sayıların okunabilir biçimde görüntülenmesi nasıl sağlanır?"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

# Yeni NumPy kullanımında tavsiye edilen yöntem:
rng = np.random.default_rng(RANDOM_STATE)
sns.set_style("whitegrid") #sns.set_theme(style=whitegrid)
sns.set_palette("Set2")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)

2.	Örnek (Sentetik) Veri Setinin Oluşturulması
1000 müşteriden oluşan örnek bir veri seti (yaş, gelir, şehir, abonelik tipi, kayıt tarihi, aylık harcama, memnuniyet puanı, churn) nasıl üretilir? Gerçek dünya verilerindeki tutarsızlıkları simüle etmek için şehir isimlerinde kasıtlı yazım farklılıkları (İstanbul/istanbul, Ankara/ANKARA) nasıl eklenir?


In [ ]:
musteri_sayisi = 1000

# -----------------------------
# 1. Temel verileri oluşturma
# -----------------------------

yas = np.random.randint(18, 70, musteri_sayisi)

gelir = np.random.randint(15000, 120000, musteri_sayisi)

sehirler = [
    "İstanbul",
    "Ankara",
    "İzmir",
    "Adana",
    "Bursa"
]

sehir = np.random.choice(sehirler, musteri_sayisi)

abonelik_tipleri = [
    "Basic",
    "Standard",
    "Premium"
]

abonelik_tipi = np.random.choice(
    abonelik_tipleri,
    musteri_sayisi,
    p=[0.45, 0.35, 0.20]
)

# Son 5 yıl içerisinden rastgele kayıt tarihi
kayit_tarihi = pd.to_datetime(
    np.random.choice(
        pd.date_range("2021-01-01", "2026-09-01"),
        musteri_sayisi
    )
)

aylik_harcama = np.round(
    np.random.uniform(100, 3000, musteri_sayisi),
    2
)

# 1-10 arasında memnuniyet puanı
memnuniyet_puani = np.random.randint(
    1,
    11,
    musteri_sayisi
)

# -----------------------------
# 2. Churn oluşturma
# -----------------------------

# Başlangıç churn olasılığı
churn_olasiligi = np.full(musteri_sayisi, 0.10)

# Memnuniyeti düşük müşterilerin churn ihtimali artsın
churn_olasiligi += np.where(
    memnuniyet_puani <= 4,
    0.30,
    0
)

# Harcaması yüksek müşterilerde biraz daha yüksek churn
churn_olasiligi += np.where(
    aylik_harcama > 2000,
    0.10,
    0
)

# Premium müşterilerde churn biraz daha düşük olsun
churn_olasiligi -= np.where(
    abonelik_tipi == "Premium",
    0.05,
    0
)

# Olasılığı 0-1 arasında tut
churn_olasiligi = np.clip(
    churn_olasiligi,
    0,
    1
)

churn = np.random.binomial(
    1,
    churn_olasiligi
)

# -----------------------------
# 3. DataFrame oluşturma
# -----------------------------

df = pd.DataFrame({
    "musteri_id": range(1, musteri_sayisi + 1),
    "yas": yas,
    "gelir": gelir,
    "sehir": sehir,
    "abonelik_tipi": abonelik_tipi,
    "kayit_tarihi": kayit_tarihi,
    "aylik_harcama": aylik_harcama,
    "memnuniyet_puani": memnuniyet_puani,
    "churn": churn
})

print(df.head())

   musteri_id  yas  gelir     sehir abonelik_tipi kayit_tarihi  aylik_harcama  \
0           1   56  49674     İzmir      Standard   2025-02-27       2,402.07   
1           2   69  50854     Bursa         Basic   2026-08-30       2,012.33   
2           3   46  61271  İstanbul      Standard   2023-08-08       1,775.64   
3           4   32  88688  İstanbul         Basic   2022-02-07       2,611.69   
4           5   60  53518     Adana         Basic   2022-08-19         939.37   

   memnuniyet_puani  churn  
0                 4      0  
1                 2      1  
2                 5      0  
3                 3      0  
4                 2      0  


3.	Kasıtlı "Kirli Veri" Enjeksiyonu
Veri setine eksik değerler (gelir, yaş, memnuniyet puanı sütunlarında) nasıl eklenir? Gelir sütununda aşırı uç değerler (outlier) nasıl oluşturulur? Yaş sütununa mantıksız değerler (-5, 150 gibi) nasıl eklenir? Veri setine kasıtlı olarak yinelenen (duplicate) satırlar nasıl eklenir?


In [ ]:
eksik_gelir_index = rng.choice(
    df.index,
    size=50,
    replace=False
)

df.loc[
    eksik_gelir_index,
    "gelir"
] = np.nan

eksik_yas_index = rng.choice(
    df.index,
    size=30,
    replace=False
)

df.loc[
    eksik_yas_index,
    "yas"
] = np.nan

eksik_memnuniyet_index = rng.choice(
    df.index,
    size=40,
    replace=False
)

df.loc[
    eksik_memnuniyet_index,
    "memnuniyet_puani"
] = np.nan


outlier_index = rng.choice(
    df.index,
    size=10,
    replace=False
)

df.loc[
    outlier_index,
    "gelir"
] = rng.integers(
    500_000,
    1_500_000,
    size=10
)

gecerli_indexler = df[
    df["yas"].notna()
].index

anomali_index = rng.choice(
    gecerli_indexler,
    size=4,
    replace=False
)

df.loc[
    anomali_index[:2],
    "yas"
] = -5

df.loc[
    anomali_index[2:],
    "yas"
] = 150

duplicate_rows = df.sample(
    n=20,
    random_state=RANDOM_STATE
)

df = pd.concat(
    [df, duplicate_rows],
    ignore_index=True
)


In [ ]:
df["sehir"].value_counts()

,count
sehir,
Adana,221
Ankara,214
Bursa,209
İstanbul,198
İzmir,178


4.	Veri Setine Genel Bakış
Veri setinin kaç satır ve kaç sütundan oluştuğu nasıl kontrol edilir? Sütunların veri tipleri ve dolu/boş hücre sayıları (.info()) nasıl incelenir? Veri setinin ilk, son ve rastgele seçilmiş birkaç satırı nasıl görüntülenir?


In [ ]:
df.shape
print("satır sayısı",df.shape[0])
print("sütun sayısı",df.shape[1])
df.dtypes
df.info
df.head()
df.tail()
df.sample(5,random_state=RANDOM_STATE)

satır sayısı 1020
sütun sayısı 9


,musteri_id,yas,gelir,sehir,abonelik_tipi,kayit_tarihi,aylik_harcama,memnuniyet_puani,churn
523,524,33.00,"19,853.00",İzmir,Standard,2021-11-06,"2,188.46",6.00,0
602,603,53.00,"23,820.00",Bursa,Standard,2025-12-12,"2,454.64",10.00,1
526,527,65.00,"36,792.00",Ankara,Premium,2024-01-01,242.74,3.00,0
31,32,44.00,"115,776.00",Ankara,Basic,2023-12-28,"2,054.13",4.00,0
616,617,41.00,"40,510.00",Bursa,Basic,2025-05-11,"2,562.38",9.00,1


5.	Eksik Değer Analizi
Her sütundaki eksik değer sayısı, benzersiz değer sayısı ve eksik değer yüzdesini gösteren bir özet tablo nasıl oluşturulur? Eksik değerlerin veri setindeki dağılımı bir ısı haritası (heatmap) ile nasıl görselleştirilir?


In [ ]:
eksik_ozet = pd.DataFrame({
    "Eksik_Sayisi": df.isnull().sum(),

    "Benzersiz_Deger": df.nunique(
        dropna=True
    ),

    "Eksik_Yuzdesi": (
        df.isnull().mean() * 100
    ).round(2)
})

eksik_ozet
eksik_ozet.sort_values(
    by="Eksik_Yuzdesi",
    ascending=False
)
plt.figure(figsize=(12, 6))

sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title(
    "Veri Setindeki Eksik Değerlerin Dağılımı"
)

plt.xlabel("Sütunlar")
plt.ylabel("Kayıtlar")

plt.show()




In [ ]:
duplicate_sayisi = df.duplicated().sum()

print(
    "Duplicate satır sayısı:",
    duplicate_sayisi
)
df[
    df.duplicated()
]
df[
    df.duplicated(
        keep=False
    )
].sort_values("musteri_id")
# df = df.drop_duplicates()

In [ ]:
df["sehir"].value_counts(
    dropna=False
)
df["sehir_temiz"] = (
    df["sehir"]
    .astype("string")
    .str.strip()
)
df["sehir_temiz"] = (
    df["sehir_temiz"]
    .str.lower()
)
df["sehir_temiz"].value_counts()

In [ ]:
anormal_yaslar = df[
    (df["yas"] < 0) |
    (df["yas"] > 100)
]

anormal_yaslar
df.loc[
    (df["yas"] < 0) |
    (df["yas"] > 100),
    [
        "musteri_id",
        "yas",
        "sehir",
        "abonelik_tipi"
    ]
]
(
    (df["yas"] < 0) |
    (df["yas"] > 100)
).sum()

In [ ]:
df.describe()
df.select_dtypes(
    include="number"
).median()
sayisal_df = df.select_dtypes(
    include="number"
)

sayisal_df.skew()
dagilim_istatistikleri = pd.DataFrame({
    "Skewness": sayisal_df.skew(),
    "Kurtosis": sayisal_df.kurtosis()
})

dagilim_istatistikleri

In [ ]:
sayisal_sutunlar = [
    "yas",
    "gelir",
    "aylik_harcama",
    "memnuniyet_puani"
]
for sutun in sayisal_sutunlar:

    plt.figure(figsize=(10, 5))

    sns.histplot(
        data=df,
        x=sutun,
        kde=True,
        bins=30
    )

    plt.title(
        f"{sutun} Dağılımı"
    )

    plt.xlabel(sutun)
    plt.ylabel("Frekans")

    plt.show()


In [ ]:
df["sehir_temiz"].value_counts(
    normalize=True,
    dropna=False
) * 100
(
    df["sehir_temiz"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)
(
    df["abonelik_tipi"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)
(
    df["memnuniyet_puani"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
    .sort_index()
)
(
    df["churn"]
    .value_counts(
        normalize=True,
        dropna=False
    )
    .mul(100)
    .round(2)
)

In [ ]:
korelasyon_sutunlari = [
    "yas",
    "gelir",
    "aylik_harcama",
    "memnuniyet_puani",
    "churn"
]

korelasyon = df[
    korelasyon_sutunlari
].corr()

korelasyon
korelasyon = df[
    korelasyon_sutunlari
].corr(
    method="pearson"
)
plt.figure(figsize=(10, 7))

sns.heatmap(
    korelasyon,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0
)

plt.title(
    "Sayısal Değişkenler Arası Korelasyon Matrisi"
)

plt.show()